# 01 – Data Exploration
First look at the SIMD 2020v2 indicators and the Data Zone (2011) Urban Rural Classification before cleaning.

In [1]:
import pandas as pd
from pathlib import Path

RAW = Path("../data/raw")

simd_file = next(RAW.glob("*indicators*.xlsx"))
ur_file = next(RAW.glob("datazone2011_urban_rural*.csv"))

print(simd_file.name)
print(pd.ExcelFile(simd_file).sheet_names)

SIMD+2020v2+-+indicators.xlsx
['Contents & notes', 'Indicator descriptions', 'Data']


In [2]:
simd = pd.read_excel(simd_file, sheet_name="Data")
ur = pd.read_csv(ur_file)

print("SIMD:", simd.shape)
print("Urban Rural:", ur.shape)
print(simd.columns.tolist())

SIMD: (6976, 37)
Urban Rural: (6976, 5)
['Data_Zone', 'Intermediate_Zone', 'Council_area', 'Total_population', 'Working_age_population', 'Income_rate', 'Income_count', 'Employment_rate', 'Employment_count', 'CIF', 'ALCOHOL', 'DRUG', 'SMR', 'DEPRESS', 'LBWT', 'EMERG', 'Attendance', 'Attainment', 'no_qualifications', 'not_participating', 'University', 'drive_petrol', 'drive_GP', 'drive_post', 'drive_primary', 'drive_retail', 'drive_secondary', 'PT_GP', 'PT_post', 'PT_retail', 'Broadband', 'crime_count', 'crime_rate', 'overcrowded_count', 'nocentralheat_count', 'overcrowded_rate', 'nocentralheat_rate']


In [3]:
simd.info()

<class 'pandas.DataFrame'>
RangeIndex: 6976 entries, 0 to 6975
Data columns (total 37 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Data_Zone               6976 non-null   str    
 1   Intermediate_Zone       6976 non-null   str    
 2   Council_area            6976 non-null   str    
 3   Total_population        6976 non-null   int64  
 4   Working_age_population  6976 non-null   int64  
 5   Income_rate             6976 non-null   object 
 6   Income_count            6976 non-null   int64  
 7   Employment_rate         6976 non-null   object 
 8   Employment_count        6976 non-null   int64  
 9   CIF                     6976 non-null   object 
 10  ALCOHOL                 6976 non-null   object 
 11  DRUG                    6976 non-null   object 
 12  SMR                     6976 non-null   object 
 13  DEPRESS                 6976 non-null   object 
 14  LBWT                    6976 non-null   object 
 15

In [4]:
print(simd.dtypes.to_string())

Data_Zone                     str
Intermediate_Zone             str
Council_area                  str
Total_population            int64
Working_age_population      int64
Income_rate                object
Income_count                int64
Employment_rate            object
Employment_count            int64
CIF                        object
ALCOHOL                    object
DRUG                       object
SMR                        object
DEPRESS                    object
LBWT                       object
EMERG                      object
Attendance                 object
Attainment                 object
no_qualifications         float64
not_participating          object
University                 object
drive_petrol              float64
drive_GP                  float64
drive_post                float64
drive_primary             float64
drive_retail              float64
drive_secondary           float64
PT_GP                     float64
PT_post                   float64
PT_retail     

In [5]:
travel_cols = [c for c in simd.columns if c.startswith(("drive_", "PT_"))]
print(travel_cols)

simd[travel_cols].describe().T

['drive_petrol', 'drive_GP', 'drive_post', 'drive_primary', 'drive_retail', 'drive_secondary', 'PT_GP', 'PT_post', 'PT_retail']


,count,mean,std,min,25%,50%,75%,max
drive_petrol,6976.0,3.873302,2.959352,0.661036,2.286015,3.171771,4.418209,64.953234
drive_GP,6976.0,3.572560,2.823331,0.560435,2.024008,2.900345,4.213584,89.376907
drive_post,6976.0,2.928204,1.757004,0.650355,1.814031,2.534832,3.465623,23.938921
drive_primary,6976.0,3.052751,1.372120,0.940951,2.215893,2.795570,3.526768,29.885949
drive_retail,6976.0,5.341315,6.000244,0.785414,2.883818,4.100013,6.041287,190.000000
drive_secondary,6976.0,6.102103,5.103198,1.017791,3.627010,4.890670,6.822313,119.231125
PT_GP,6976.0,10.469649,6.292247,1.595565,6.432432,8.958820,12.652174,119.789958
PT_post,6976.0,8.873081,4.741516,1.831674,5.722540,7.722572,10.621549,41.010487
PT_retail,6976.0,13.546196,10.709884,1.835432,7.982432,11.246941,16.059470,190.000000


In [6]:
capped = simd[(simd["drive_retail"] >= 190) | (simd["PT_retail"] >= 190)]

print(len(capped), "zones")
capped[["Data_Zone", "Intermediate_Zone", "Council_area", "Total_population", "drive_retail", "PT_retail"]]

5 zones


,Data_Zone,Intermediate_Zone,Council_area,Total_population,drive_retail,PT_retail
5325,S01011831,Isles,Orkney Islands,845,190.000000,190.0
5326,S01011832,Isles,Orkney Islands,808,146.469457,190.0
5908,S01012414,North and East Isles,Shetland Islands,700,121.021573,190.0
5909,S01012415,North and East Isles,Shetland Islands,598,74.787235,190.0
5910,S01012416,North and East Isles,Shetland Islands,506,93.008873,190.0


### Key observations – SIMD indicators
- 6,976 data zones in both files, one row per zone; Council_area is already included.
- The data is on the "Data" sheet; the first two sheets are notes.
- 16 columns are stored as text because of "*" (suppressed values). None of these are the travel columns.
- All 9 travel time columns (minutes) are numeric with no missing values.
- Times are right-skewed (mean > median), so medians are used for comparisons.
- Median times: GP 2.9 min by car vs 9.0 min by public transport; post office 2.5 vs 7.7; retail 4.1 vs 11.2.
- 5 island zones in Orkney and Shetland (~3,460 people) have PT_retail = 190, which looks like a cap.
  They will be kept and flagged rather than removed.

In [7]:
print(ur.dtypes)
ur["UrbanRural6fold2020"].value_counts().sort_index()

DataZone                 str
UrbanRural2fold2020    int64
UrbanRural3fold2020    int64
UrbanRural6fold2020    int64
UrbanRural8fold2020    int64
dtype: object


UrbanRural6fold2020
1    2496
2    2459
3     613
4     202
5     805
6     401
Name: count, dtype: int64

### Key observations – Urban Rural file
- 6,976 zones, codes stored as integers; the 6-fold categories sum to 6,976.
- Codes: 1 Large Urban, 2 Other Urban, 3 Accessible Small Towns, 4 Remote Small Towns,
  5 Accessible Rural, 6 Remote Rural. Labels will be added in cleaning.
- Data zones have similar populations, so rural zones are fewer (17%) but cover far more land.
- The key column is `DataZone` here and `Data_Zone` in SIMD.

In [8]:
missing_in_ur = ~simd["Data_Zone"].isin(ur["DataZone"])
print("SIMD zones not in Urban Rural file:", missing_in_ur.sum())
print("Duplicate codes in Urban Rural file:", ur["DataZone"].duplicated().sum())

SIMD zones not in Urban Rural file: 0
Duplicate codes in Urban Rural file: 0


### Join check
All 6,976 SIMD data zones exist in the Urban Rural file, with no duplicate codes,
confirming both use 2011 data zones.

## Summary
Cleaning will need to:
- keep only the columns needed (identifiers, population, travel times);
- join the Urban Rural classification and add readable category labels;
- calculate the public transport vs car gap for each service;
- flag the 5 island zones where PT_retail is capped at 190 minutes.